# CFIS V2 histórica — Phishing | Benchmark científico (5 splits)

A implementação binária de Permutation Importance abaixo foi **transcrita sem alterações da célula histórica 9** do projeto CFIS (`Untitled49.ipynb`), e não é uma implementação nova apenas semelhante. Antes do benchmark, o notebook verifica igualdade numérica com `sklearn.inspection.permutation_importance` sob a mesma semente, `scoring="accuracy"` e 10 repetições. A formulação híbrida mantém três componentes com pesos iguais; a normalização deve ser conferida contra o CFIS histórico integral antes de afirmar equivalência de ponta a ponta. O CSV do dataset não está incluído neste arquivo.

In [2]:
# Bibliotecas disponíveis no Colab: numpy pandas scipy scikit-learn matplotlib seaborn psutil
import os, time, gc, warnings, platform, json, itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr, kendalltau, friedmanchisquare, wilcoxon
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from scipy.special import expit
try:
 import psutil
except ImportError:
 psutil = None
SEED=42; N_SPLITS=5; N_REPEATS=10; os.makedirs('resultados_cfis',exist_ok=True)
np.random.seed(SEED)
print('Python:',platform.python_version(),'| CPU:',os.cpu_count(),'| psutil:',psutil is not None)

from sklearn.utils import check_random_state


Python: 3.13.18 | CPU: 2 | psutil: True


## 1. Carregamento e auditoria do dataset
O artigo descreve 87 atributos após excluir a URL; o notebook confere a quantidade real. `status`: `legitimate=0`, `phishing=1`. Ausências e infinitos são tratados **dentro do treinamento de cada divisão**.

In [3]:
PATH='dataset_phishing.csv'
if not os.path.exists(PATH):
 try:
  from google.colab import files
  print('Envie dataset_phishing.csv'); files.upload()
 except ImportError: pass
if not os.path.exists(PATH):
 raise FileNotFoundError('Envie dataset_phishing.csv para executar o benchmark; nenhum resultado foi inventado.')
df=pd.read_csv(PATH)
assert 'status' in df.columns, 'Coluna status não encontrada'
y=df['status'].map({'legitimate':0,'phishing':1}) if df['status'].dtype=='object' else df['status'].astype(int)
assert y.notna().all() and set(y.unique())=={0,1}, 'Verifique codificação binária do alvo'
X=df.drop(columns=['status','url'],errors='ignore').apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
print('Registros:',len(X),'| Características:',X.shape[1],'| Classes:',y.value_counts().to_dict())
print('Duplicatas exatas:',int(df.duplicated().sum()),'| Valores ausentes:',int(X.isna().sum().sum()))
print('Constantes:',X.columns[X.nunique(dropna=True)<=1].tolist())
print('Maior ausência (%):'); display((100*X.isna().mean()).sort_values(ascending=False).head(10).to_frame('missing_%'))
print('Correlação de Pearson (descrição, não seleção):'); display(X.corrwith(y).sort_values().iloc[np.r_[0:5,-5:0]].to_frame('rho'))


Envie dataset_phishing.csv


Saving dataset_phishing.csv to dataset_phishing.csv
Registros: 11430 | Características: 87 | Classes: {0: 5715, 1: 5715}
Duplicatas exatas: 0 | Valores ausentes: 0
Constantes: ['nb_or', 'ratio_nullHyperlinks', 'ratio_intRedirection', 'ratio_intErrors', 'submit_email', 'sfh']
Maior ausência (%):


,missing_%
length_url,0.0
length_hostname,0.0
ip,0.0
nb_dots,0.0
nb_hyphens,0.0
nb_at,0.0
nb_qm,0.0
nb_and,0.0
nb_or,0.0
nb_eq,0.0


Correlação de Pearson (descrição, não seleção):


/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,rho
page_rank,-0.511137
nb_www,-0.443468
nb_hyperlinks,-0.342628
domain_age,-0.331889
ratio_intHyperlinks,-0.243982
ratio_nullHyperlinks,NaN
ratio_intRedirection,NaN
ratio_intErrors,NaN
submit_email,NaN
sfh,NaN


## 2. CFIS com Permutation Importance V2 histórica

A função `fast_binary_permutation_importance_v2` é a implementação recuperada **literalmente** do experimento histórico de otimização. Ela reutiliza `decision_function` da regressão logística e atualiza o escore após cada embaralhamento cumulativo da coluna, seguindo o gerador e a semente interna do scikit-learn. **Não usa GPU/PyTorch** nessa etapa. A função `cfis_fit` combina a PFI com coeficientes absolutos e correlação absoluta; `rho_signed` e `direction` são relatados separadamente. A normalização por soma é a mesma deste notebook anterior, mas sua equivalência à função CFIS integral do artigo deve ser auditada antes da redação final.

In [5]:
def safe_norm(a):
 a=np.asarray(a,dtype=float);a=np.nan_to_num(a,nan=0,posinf=0,neginf=0)
 a=np.maximum(a,0); return a/a.sum() if a.sum()>1e-12 else np.zeros_like(a)

# Extraído sem alterações da célula histórica 9
def fast_binary_permutation_importance_v2(
    estimator, X, y, n_repeats=10, random_state=42
):
    X = np.asarray(X)
    y = np.asarray(y)

    n_samples, n_features = X.shape
    coef = estimator.coef_[0]
    classes = estimator.classes_

    base_scores = estimator.decision_function(X)
    baseline_accuracy = np.mean(estimator.predict(X) == y)

    # Mesma semente interna utilizada pelo scikit-learn.
    rng = check_random_state(random_state)
    internal_seed = rng.randint(np.iinfo(np.int32).max)

    importances = np.empty(
        (n_features, n_repeats), dtype=float
    )

    for j in range(n_features):

        feature_rng = check_random_state(internal_seed)
        shuffle_idx = np.arange(n_samples)

        # A coluna é modificada cumulativamente.
        current_column = X[:, j].copy()
        original_column = X[:, j]

        for r in range(n_repeats):

            feature_rng.shuffle(shuffle_idx)

            # CORREÇÃO: permutar a coluna já modificada.
            current_column = current_column[shuffle_idx].copy()

            updated_scores = (
                base_scores
                + (current_column - original_column) * coef[j]
            )

            predictions = np.where(
                updated_scores > 0,
                classes[1],
                classes[0]
            )

            accuracy = np.mean(predictions == y)

            importances[j, r] = (
                baseline_accuracy - accuracy
            )

    return importances

def pfi_fast_logreg(model, Xs, y, repeats=10, seed=42):
 """Usa exatamente a PFI V2 histórica; retorna médias das 10 permutações."""
 return fast_binary_permutation_importance_v2(model, np.asarray(Xs), np.asarray(y), n_repeats=repeats, random_state=seed).mean(axis=1)

def cfis_fit(Xtrain,ytrain,repeats=N_REPEATS,seed=SEED):
 """Retorna ranking e custo; PFI V2 histórica, apenas dados de treinamento."""
 t0=time.perf_counter(); cpu0=time.process_time();
 rss0=psutil.Process().memory_info().rss if psutil else np.nan
 model=LogisticRegression(max_iter=1500,solver='lbfgs',random_state=seed)
 model.fit(Xtrain,ytrain)
 perm=pfi_fast_logreg(model,Xtrain,ytrain,repeats=repeats,seed=seed)
 coef=model.coef_[0]
 a=np.asarray(Xtrain,dtype=float); yc=np.asarray(ytrain,dtype=float)
 ac=a-a.mean(axis=0); yc=yc-yc.mean()
 den=np.sqrt(np.sum(ac*ac,axis=0)*np.sum(yc*yc))
 rho=np.divide(ac.T@yc,den,out=np.zeros(a.shape[1]),where=den>1e-12)
 perm_n=safe_norm(perm); coef_n=safe_norm(np.abs(coef)); rho_n=safe_norm(np.abs(rho))
 score=(perm_n+coef_n+rho_n)/3
 names=list(Xtrain.columns) if hasattr(Xtrain,'columns') else [f'f{i}' for i in range(len(score))]
 rank=pd.DataFrame({'feature':names,'cfis':score,'pfi_raw':perm,'pfi_component':perm_n,
                    'coef_signed':coef,'coef_component':coef_n,'rho_signed':rho,'rho_component':rho_n})
 rank['direction']=np.where(rank.rho_signed>0,'positiva',np.where(rank.rho_signed<0,'negativa','nula'))
 rank=rank.sort_values(['cfis','feature'],ascending=[False,True]).reset_index(drop=True)
 return rank, {'selection_s':time.perf_counter()-t0,'selection_cpu_s':time.process_time()-cpu0,
               'rss_delta_mb':((psutil.Process().memory_info().rss-rss0)/2**20) if psutil else np.nan}

def prepare(Xtr,Xva):
 med=Xtr.median().fillna(0)
 a=Xtr.fillna(med);v=Xva.fillna(med)
 scaler=StandardScaler().fit(a)
 return pd.DataFrame(scaler.transform(a),columns=Xtr.columns,index=Xtr.index),pd.DataFrame(scaler.transform(v),columns=Xtr.columns,index=Xva.index)


In [ ]:
# Auditoria obrigatória: identidade numérica da PFI V2 histórica vs scikit-learn.
# O teste impede a continuação em caso de divergência.
rng=np.random.default_rng(12)
a=rng.normal(size=(120,8))
yy=(a[:,0]-a[:,2]+0.3*rng.normal(size=120)>0).astype(int)
m=LogisticRegression(max_iter=1000).fit(a,yy)
for seed in (42, 77):
    for repeats in (1, 5, 10):
        fast=fast_binary_permutation_importance_v2(m,a,yy,n_repeats=repeats,random_state=seed)
        sk=permutation_importance(m,a,yy,n_repeats=repeats,random_state=seed,scoring='accuracy',n_jobs=1).importances
        np.testing.assert_allclose(fast,sk,rtol=0,atol=1e-12,
            err_msg=f'PFI V2 difere do sklearn: seed={seed}, repeats={repeats}')
print('AUDITORIA PFI V2: PASSOU — importâncias por repetição equivalentes ao scikit-learn.')


AUDITORIA PFI V2: PASSOU — importâncias por repetição equivalentes ao scikit-learn.


**Interpretação da auditoria:** a igualdade testada é a da *Permutation Importance* binária sob acurácia, não uma certificação automática de equivalência da implementação integral do CFIS, de seus pré-processamentos ou dos resultados externos. O benchmark depende do arquivo `dataset_phishing.csv` e não contém resultados inventados.

# BENCHMARK FINAL

In [13]:

# Instalar somente as dependências ausentes
import importlib.util
import subprocess
import sys

packages = {
    "skrebate": "skrebate==0.8.4",
    "arfs": "arfs",
    "shap_select": "shap-select",
    "shap": "shap==0.54.1"
}

missing = [
    package
    for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )

from skrebate import ReliefF
from arfs.feature_selection import Leshy
from shap_select import shap_select
import shap

print("Dependências instaladas com sucesso.")
print("Agora execute novamente a B9.")


/usr/local/lib/python3.13/dist-packages/arfs/utils.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Dependências instaladas com sucesso.
Agora execute novamente a B9.


In [14]:

# ============================================================
# CFIS PHISHING — B9
# RESTAURAÇÃO DOS SELETORES APÓS REINICIALIZAÇÃO
# ============================================================

import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

from skrebate import ReliefF
from arfs.feature_selection import Leshy
from shap_select import shap_select
import shap

EPS = 1e-12

# ------------------------------------------------------------
# 1. CFS — restaurar implementação B3.1
# ------------------------------------------------------------

def rank_cfs(X_train, y_train, seed=42):

    X_np = np.asarray(X_train, dtype=np.float64)
    y_np = np.asarray(y_train, dtype=np.float64).ravel()

    names = X_train.columns.tolist()
    n_features = X_np.shape[1]

    if not np.isfinite(X_np).all():
        raise ValueError("CFS recebeu valores não finitos.")

    if not np.isfinite(y_np).all():
        raise ValueError("CFS recebeu rótulos não finitos.")

    X_centered = X_np - X_np.mean(axis=0)
    y_centered = y_np - y_np.mean()

    x_norm = np.linalg.norm(X_centered, axis=0)
    y_norm = np.linalg.norm(y_centered)

    safe_x_norm = np.where(x_norm > EPS, x_norm, 1.0)

    X_unit = X_centered / safe_x_norm

    if y_norm <= EPS:
        raise ValueError("Classe constante no treinamento.")

    y_unit = y_centered / y_norm

    relevance = np.abs(X_unit.T @ y_unit)
    relevance[x_norm <= EPS] = 0.0

    corr = np.abs(X_unit.T @ X_unit)
    corr = np.clip(corr, 0.0, 1.0)

    np.fill_diagonal(corr, 1.0)

    selected = []
    remaining = list(range(n_features))

    sum_relevance = 0.0
    sum_pair_corr = 0.0

    while remaining:

        best_feature = None
        best_merit = -np.inf

        for j in remaining:

            k = len(selected) + 1

            candidate_relevance = (
                sum_relevance + relevance[j]
            )

            candidate_pairs = (
                sum_pair_corr
                + (
                    float(corr[j, selected].sum())
                    if selected else 0.0
                )
            )

            denominator = np.sqrt(
                k + 2.0 * candidate_pairs
            )

            merit = (
                candidate_relevance
                / max(denominator, EPS)
            )

            if merit > best_merit:
                best_merit = merit
                best_feature = j

        sum_relevance += relevance[best_feature]

        if selected:
            sum_pair_corr += float(
                corr[best_feature, selected].sum()
            )

        selected.append(best_feature)
        remaining.remove(best_feature)

    return [names[j] for j in selected]


# ------------------------------------------------------------
# 2. Utilitários
# ------------------------------------------------------------

def ranking_from_scores(scores, columns):

    scores = np.asarray(scores, dtype=float).ravel()

    if len(scores) != len(columns):
        raise ValueError("Dimensão incorreta dos scores.")

    return (
        pd.Series(scores, index=columns)
        .sort_values(ascending=False, kind="stable")
        .index.tolist()
    )


def benchmark_discrete_mask(X_train):

    return np.array([
        X_train[c].nunique(dropna=False) <= 2
        for c in X_train.columns
    ], dtype=bool)


def reference_model(seed):

    return RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=1
    )


# ------------------------------------------------------------
# 3. Adaptadores existentes
# ------------------------------------------------------------

def selector_cfis(X_train, y_train, seed, k):

    rank, cost = cfis_fit(
        X_train, y_train, seed=seed
    )

    return rank["feature"].tolist()[:k]


def selector_l1(X_train, y_train, seed, k):

    model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=0.1,
        max_iter=1500,
        random_state=seed
    )

    model.fit(X_train, y_train)

    return ranking_from_scores(
        np.abs(model.coef_[0]),
        X_train.columns
    )[:k]


def selector_mi(X_train, y_train, seed, k):

    scores = mutual_info_classif(
        X_train,
        y_train,
        discrete_features=benchmark_discrete_mask(X_train),
        random_state=seed
    )

    return ranking_from_scores(
        scores,
        X_train.columns
    )[:k]


def selector_cfs(X_train, y_train, seed, k):

    return rank_cfs(
        X_train, y_train, seed=seed
    )[:k]


def selector_relief(X_train, y_train, seed, k):

    model = ReliefF(
        n_features_to_select=k,
        n_neighbors=10,
        n_jobs=1
    )

    model.fit(
        X_train.to_numpy(dtype=float),
        np.asarray(y_train).ravel()
    )

    return ranking_from_scores(
        model.feature_importances_,
        X_train.columns
    )[:k]


def selector_shap(X_train, y_train, seed, k):

    model = reference_model(seed)
    model.fit(X_train, y_train)

    sample = X_train.sample(
        n=min(500, len(X_train)),
        random_state=seed
    )

    values = shap.TreeExplainer(
        model
    ).shap_values(sample)

    if isinstance(values, list):
        values = values[1]

    values = np.asarray(values)

    if values.ndim == 3:
        values = values[:, :, 1]

    scores = np.mean(np.abs(values), axis=0)

    return ranking_from_scores(
        scores,
        X_train.columns
    )[:k]


def selector_pfi(X_train, y_train, seed, k):

    X_fit, X_eval, y_fit, y_eval = train_test_split(
        X_train,
        y_train,
        test_size=0.25,
        stratify=y_train,
        random_state=seed
    )

    model = reference_model(seed)
    model.fit(X_fit, y_fit)

    result = permutation_importance(
        model,
        X_eval,
        y_eval,
        n_repeats=10,
        random_state=seed,
        scoring="f1_macro",
        n_jobs=1
    )

    return ranking_from_scores(
        result.importances_mean,
        X_train.columns
    )[:k]


def selector_random(X_train, y_train, seed, k):

    rng = np.random.default_rng(seed)

    return rng.permutation(
        X_train.columns.to_numpy()
    ).tolist()[:k]


# ------------------------------------------------------------
# 4. Leshy e SHAP-Select
# ------------------------------------------------------------

# Não restauramos a configuração custosa do Leshy.
# O método será definido após escolhermos um orçamento
# computacional apropriado e documentado.

def selector_shap_select(X_train, y_train, seed, k):

    X_fit, X_internal, y_fit, y_internal = train_test_split(
        X_train,
        y_train,
        test_size=0.25,
        stratify=y_train,
        random_state=seed
    )

    model = reference_model(seed)
    model.fit(X_fit, y_fit)

    result = shap_select(
        tree_model=model,
        validation_df=X_internal,
        target=pd.Series(
            np.asarray(y_internal).ravel(),
            index=X_internal.index
        ),
        feature_names=X_train.columns.tolist(),
        task="binary",
        threshold=0.05
    )

    if "Selected" not in result.columns:
        raise ValueError(
            "SHAP-Select não retornou coluna Selected."
        )

    feature_col = next(
        (
            col for col in result.columns
            if str(col).lower() in [
                "feature",
                "features",
                "feature_name",
                "feature_names"
            ]
        ),
        None
    )

    if feature_col is not None:
        return result.loc[
            result["Selected"].astype(bool),
            feature_col
        ].tolist()

    return result.index[
        result["Selected"].astype(bool)
    ].tolist()


# ------------------------------------------------------------
# 5. Registro dos seletores
# ------------------------------------------------------------

BENCHMARK_SELECTORS = {
    "CFIS": selector_cfis,
    "L1": selector_l1,
    "MutualInfo": selector_mi,
    "CFS": selector_cfs,
    "ReliefF": selector_relief,
    "TreeSHAP": selector_shap,
    "PFI": selector_pfi,
    "RandomTopK": selector_random,
    "SHAP-Select": selector_shap_select
}

print("=" * 70)
print("B9 — RESTAURAÇÃO CONCLUÍDA")
print("=" * 70)

for name in BENCHMARK_SELECTORS:
    print(" -", name)

print("\nSeletores restaurados:", len(BENCHMARK_SELECTORS))
print("Leshy: aguardando definição experimental")
print("CFIS-V2: implementação original preservada")
print("Nenhum treinamento executado.")


B9 — RESTAURAÇÃO CONCLUÍDA
 - CFIS
 - L1
 - MutualInfo
 - CFS
 - ReliefF
 - TreeSHAP
 - PFI
 - RandomTopK
 - SHAP-Select

Seletores restaurados: 9
Leshy: aguardando definição experimental
CFIS-V2: implementação original preservada
Nenhum treinamento executado.


In [15]:

# ============================================================
# B12.1 — VERIFICAR FUNÇÕES NECESSÁRIAS
# ============================================================

required = [
    "cfis_fit",
    "rank_cfs",
    "BENCHMARK_SPLITS",
    "BENCHMARK_SELECTORS",
    "prepare",
    "reference_model",
    "models",
    "Xdev",
    "ydev",
    "SEED",
    "BENCHMARK_K",
    "PASTAS"
]

print("VERIFICAÇÃO DO AMBIENTE\n")

missing = []

for name in required:
    exists = name in globals()
    print(f"{name:24s}: {'OK' if exists else 'AUSENTE'}")

    if not exists:
        missing.append(name)

print("\nTotal de objetos ausentes:", len(missing))

if missing:
    print("\nÉ necessário restaurar:")
    for name in missing:
        print(" -", name)
else:
    print("\nAmbiente pronto para executar a B12.")


VERIFICAÇÃO DO AMBIENTE

cfis_fit                : OK
rank_cfs                : OK
BENCHMARK_SPLITS        : OK
BENCHMARK_SELECTORS     : OK
prepare                 : OK
reference_model         : OK
models                  : OK
Xdev                    : OK
ydev                    : OK
SEED                    : OK
BENCHMARK_K             : OK
PASTAS                  : OK

Total de objetos ausentes: 0

Ambiente pronto para executar a B12.


In [10]:

from google.colab import drive
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
import json

# Montar Drive automaticamente
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/CFIS_Phishing_Benchmark_Final")

if not ROOT.is_dir():
    print("Pastas disponíveis no MyDrive:")
    print([p.name for p in Path("/content/drive/MyDrive").iterdir()])
    raise FileNotFoundError("Pasta CFIS_Phishing_Benchmark_Final não encontrada.")

PASTAS = {
    "protocolo": ROOT / "01_Dataset_Protocolo",
    "codigo": ROOT / "02_Notebooks_Codigo",
    "folds": ROOT / "03_Resultados_por_Fold",
    "rankings": ROOT / "04_Rankings_Estabilidade",
    "estatistica": ROOT / "05_Eficiencia_Estatistica",
    "artigo": ROOT / "06_Tabelas_Graficos_Artigo"
}

with open(PASTAS["protocolo"] / "protocolo_benchmark.json", encoding="utf-8") as f:
    protocolo = json.load(f)

SEED = int(protocolo["seed"])
N_SPLITS = int(protocolo["n_splits"])
BENCHMARK_K = protocolo["k_por_classificador"]

BENCHMARK_SPLITS = list(
    StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED
    ).split(Xdev, ydev)
)

def reference_model(seed):
    return RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=1
    )

print("Drive montado:", ROOT)
print("Folds recuperados:", len(BENCHMARK_SPLITS))
print("CFIS original:", "OK" if callable(globals().get("cfis_fit")) else "AUSENTE")
print("Checkpoint piloto:", (PASTAS["folds"] / "b63_piloto_fold1.csv").exists())
print("Configurações restauradas.")


Mounted at /content/drive
Drive montado: /content/drive/MyDrive/CFIS_Phishing_Benchmark_Final
Folds recuperados: 5
CFIS original: OK
Checkpoint piloto: True
Configurações restauradas.


In [18]:

import warnings
import logging
import os

# Ignorar warnings do Python
warnings.filterwarnings("ignore")

# Reduzir mensagens de bibliotecas
logging.getLogger("shap").setLevel(logging.ERROR)
logging.getLogger("sklearn").setLevel(logging.ERROR)

# Reduzir logs do TensorFlow, caso utilizado
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

print("Warnings desativados.")

# ============================================================
# CFIS-V2 — B12.3 | BENCHMARK PRINCIPAL
# 8 seletores + All Features
# 5 folds x 3 classificadores = 135 avaliações
# ============================================================

import os, time, json, traceback, threading
from pathlib import Path
import numpy as np
import pandas as pd
import psutil

from sklearn.base import clone
from sklearn.metrics import (
    f1_score, balanced_accuracy_score,
    roc_auc_score, matthews_corrcoef,
    r2_score, mean_squared_error
)

METHODS = [
    "CFIS", "L1", "MutualInfo", "CFS",
    "ReliefF", "TreeSHAP", "PFI",
    "RandomTopK", "All Features"
]

CLASSIFIERS = [
    "DecisionTree", "LogReg", "RandomForest"
]

RESULT_FILE = (
    PASTAS["folds"] /
    "benchmark_principal_5fold.csv"
)

FEATURE_FILE = (
    PASTAS["rankings"] /
    "benchmark_principal_features.jsonl"
)

for folder in PASTAS.values():
    folder.mkdir(parents=True, exist_ok=True)

required = [
    "cfis_fit", "rank_cfs", "BENCHMARK_SPLITS",
    "BENCHMARK_SELECTORS", "prepare",
    "models", "Xdev", "ydev", "BENCHMARK_K"
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Objetos ausentes: {missing}")

# ------------------------------------------------------------
# Monitor de recursos
# ------------------------------------------------------------

class ResourceMonitor:
    def __init__(self):
        self.process = psutil.Process()
        self.running = False
        self.peak_rss = 0
        self.cpu_start = None
        self.thread = None

    def _sample(self):
        while self.running:
            try:
                rss = self.process.memory_info().rss
                self.peak_rss = max(self.peak_rss, rss)
            except psutil.Error:
                pass
            time.sleep(0.05)

    def start(self):
        self.peak_rss = self.process.memory_info().rss
        self.cpu_start = self.process.cpu_times()
        self.running = True
        self.thread = threading.Thread(
            target=self._sample, daemon=True
        )
        self.thread.start()

    def stop(self):
        self.running = False
        self.thread.join(timeout=2)
        cpu_end = self.process.cpu_times()
        cpu_seconds = (
            cpu_end.user + cpu_end.system
            - self.cpu_start.user - self.cpu_start.system
        )
        return {
            "cpu_time_s": cpu_seconds,
            "peak_rss_mib": self.peak_rss / (1024 ** 2)
        }

# ------------------------------------------------------------
# Carregar checkpoint
# ------------------------------------------------------------

if RESULT_FILE.exists():
    results = pd.read_csv(RESULT_FILE)
else:
    results = pd.DataFrame()

completed = set()

if not results.empty:
    completed = {
        (int(r.fold), r.classifier, r.method)
        for r in results.itertuples()
        if r.status == "OK"
    }

def save_checkpoint(record):
    global results

    if not results.empty:
        mask = (
            (results["fold"] == record["fold"])
            & (results["classifier"] == record["classifier"])
            & (results["method"] == record["method"])
        )
        results = results.loc[~mask].copy()

    results = pd.concat(
        [results, pd.DataFrame([record])],
        ignore_index=True
    )

    tmp = RESULT_FILE.with_suffix(".tmp")
    results.to_csv(tmp, index=False)
    os.replace(tmp, RESULT_FILE)

# ------------------------------------------------------------
# Benchmark
# ------------------------------------------------------------

total = (
    len(BENCHMARK_SPLITS)
    * len(CLASSIFIERS)
    * len(METHODS)
)

print("=" * 70)
print("CFIS-V2 — BENCHMARK PRINCIPAL")
print("Total:", total)
print("Já concluídos:", len(completed))
print("=" * 70, flush=True)

counter = 0

for fold, (train_idx, val_idx) in enumerate(
    BENCHMARK_SPLITS, start=1
):

    X_train, X_val = prepare(
        Xdev.iloc[train_idx].copy(),
        Xdev.iloc[val_idx].copy()
    )

    y_train = ydev.iloc[train_idx].copy()
    y_val = ydev.iloc[val_idx].copy()

    for classifier in CLASSIFIERS:

        k = int(BENCHMARK_K[classifier])

        for method in METHODS:

            counter += 1
            key = (fold, classifier, method)

            if key in completed:
                continue

            print(
                f"[{counter}/{total}] "
                f"Fold {fold} | {classifier} | {method}",
                flush=True
            )

            monitor = ResourceMonitor()
            start_total = time.perf_counter()
            monitor.start()

            record = {
                "fold": fold,
                "classifier": classifier,
                "method": method,
                "k_reference": k,
                "status": "ERROR",
                "error": ""
            }

            try:
                seed = SEED + fold - 1

                start_selection = time.perf_counter()

                if method == "All Features":
                    selected = X_train.columns.tolist()
                else:
                    selected = BENCHMARK_SELECTORS[method](
                        X_train, y_train, seed, k
                    )

                selection_time = (
                    time.perf_counter() - start_selection
                )

                selected = list(selected)

                if len(selected) != len(set(selected)):
                    raise ValueError(
                        "Características duplicadas"
                    )

                unknown = (
                    set(selected) - set(X_train.columns)
                )

                if unknown:
                    raise ValueError(
                        f"Características desconhecidas: {unknown}"
                    )

                if not selected:
                    raise ValueError(
                        "Seleção retornou conjunto vazio"
                    )

                selected_set = set(selected)
                columns = [
                    c for c in X_train.columns
                    if c in selected_set
                ]

                if method != "All Features" and len(columns) != k:
                    raise ValueError(
                        f"Top-K incorreto: {len(columns)} != {k}"
                    )

                clf = clone(models[classifier])

                start_train = time.perf_counter()
                clf.fit(X_train[columns], y_train)
                train_time = (
                    time.perf_counter() - start_train
                )

                start_predict = time.perf_counter()
                pred = clf.predict(X_val[columns])

                if hasattr(clf, "predict_proba"):
                    proba = clf.predict_proba(
                        X_val[columns]
                    )[:, 1]
                else:
                    proba = clf.decision_function(
                        X_val[columns]
                    )

                predict_time = (
                    time.perf_counter() - start_predict
                )

                record.update({
                    "k_selected": len(columns),
                    "selection_mode": (
                        "all" if method == "All Features"
                        else "top_k"
                    ),
                    "macro_f1": f1_score(
                        y_val, pred, average="macro"
                    ),
                    "balanced_accuracy":
                        balanced_accuracy_score(y_val, pred),
                    "roc_auc": roc_auc_score(y_val, proba),
                    "mcc": matthews_corrcoef(y_val, pred),
                    "r2_labels": r2_score(y_val, pred),
                    "rmse_labels": np.sqrt(
                        mean_squared_error(y_val, pred)
                    ),
                    "selection_time_s": selection_time,
                    "train_time_s": train_time,
                    "predict_time_s": predict_time,
                    "status": "OK",
                    "error": ""
                })

                # Persistir características por execução
                with open(
                    FEATURE_FILE, "a", encoding="utf-8"
                ) as f:
                    f.write(
                        json.dumps({
                            "fold": fold,
                            "classifier": classifier,
                            "method": method,
                            "k_selected": len(columns),
                            "features": columns
                        }, ensure_ascii=False) + "\n"
                    )

            except Exception as exc:
                record["error"] = (
                    f"{type(exc).__name__}: {exc}"
                )
                print(
                    "  ERRO:", record["error"],
                    flush=True
                )

            finally:
                resources = monitor.stop()
                record.update(resources)
                record["total_time_s"] = (
                    time.perf_counter() - start_total
                )

                save_checkpoint(record)

                if record["status"] == "OK":
                    completed.add(key)
                    print(
                        f"  OK | F1={record['macro_f1']:.4f}"
                        f" | K={record['k_selected']}"
                        f" | Total={record['total_time_s']:.2f}s",
                        flush=True
                    )

print("\n" + "=" * 70)
print("BENCHMARK PRINCIPAL ENCERRADO")
print("=" * 70)

print(results["status"].value_counts())

summary = (
    results[results["status"] == "OK"]
    .groupby(["classifier", "method"])
    .agg(
        folds=("macro_f1", "count"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        balanced_accuracy_mean=(
            "balanced_accuracy", "mean"
        ),
        r2_mean=("r2_labels", "mean"),
        rmse_mean=("rmse_labels", "mean"),
        selection_time_mean=(
            "selection_time_s", "mean"
        ),
        cpu_time_mean=("cpu_time_s", "mean"),
        peak_rss_mean=("peak_rss_mib", "mean"),
        total_time_mean=("total_time_s", "mean")
    )
    .reset_index()
)

display(summary)

summary.to_csv(
    PASTAS["artigo"] /
    "benchmark_principal_resumo.csv",
    index=False
)

print("\nCheckpoint:", RESULT_FILE)
print("Características:", FEATURE_FILE)


Warnings desativados.
CFIS-V2 — BENCHMARK PRINCIPAL
Total: 135
Já concluídos: 4
[5/135] Fold 1 | DecisionTree | ReliefF
  OK | F1=0.9138 | K=15 | Total=69.28s
[6/135] Fold 1 | DecisionTree | TreeSHAP
  OK | F1=0.9219 | K=15 | Total=20.79s
[7/135] Fold 1 | DecisionTree | PFI
  OK | F1=0.9332 | K=15 | Total=30.49s
[8/135] Fold 1 | DecisionTree | RandomTopK
  OK | F1=0.8325 | K=15 | Total=0.05s
[9/135] Fold 1 | DecisionTree | All Features
  OK | F1=0.9313 | K=87 | Total=0.20s
[10/135] Fold 1 | LogReg | CFIS
  OK | F1=0.9444 | K=60 | Total=0.62s
[11/135] Fold 1 | LogReg | L1
  OK | F1=0.9438 | K=60 | Total=0.21s
[12/135] Fold 1 | LogReg | MutualInfo
  OK | F1=0.9388 | K=60 | Total=1.67s
[13/135] Fold 1 | LogReg | CFS
  OK | F1=0.9357 | K=60 | Total=0.25s
[14/135] Fold 1 | LogReg | ReliefF
  OK | F1=0.9407 | K=60 | Total=63.67s
[15/135] Fold 1 | LogReg | TreeSHAP
  OK | F1=0.9438 | K=60 | Total=19.82s
[16/135] Fold 1 | LogReg | PFI
  OK | F1=0.9413 | K=60 | Total=30.30s
[17/135] Fold 1 | Lo

,classifier,method,folds,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,r2_mean,rmse_mean,selection_time_mean,cpu_time_mean,peak_rss_mean,total_time_mean
0,DecisionTree,All Features,5,0.929879,0.004646,0.929885,0.719534,0.264678,0.000023,0.186,493.121875,0.223293
1,DecisionTree,CFIS,5,0.930756,0.006123,0.930758,0.723032,0.262936,0.744633,1.066,489.744531,0.903262
2,DecisionTree,CFS,5,0.925878,0.009629,0.925886,0.703542,0.271761,0.134435,0.262,490.250781,0.260565
3,DecisionTree,L1,5,0.929505,0.005079,0.929509,0.718032,0.265365,0.141545,0.188,500.941406,0.247395
4,DecisionTree,MutualInfo,5,0.916503,0.007241,0.916510,0.666043,0.288722,2.337852,2.086,490.174219,2.530450
5,DecisionTree,PFI,5,0.927254,0.009127,0.927259,0.709033,0.269301,30.750615,30.554,493.113281,30.848564
6,DecisionTree,RandomTopK,5,0.824615,0.046945,0.824649,0.298584,0.415772,0.000306,0.046,493.114062,0.072353
7,DecisionTree,ReliefF,5,0.919880,0.005424,0.919885,0.679543,0.282914,65.357620,64.320,1695.242188,65.456136
8,DecisionTree,TreeSHAP,5,0.924132,0.005026,0.924135,0.696539,0.275315,19.843089,19.600,493.109375,19.934274
9,LogReg,All Features,5,0.945506,0.002303,0.945507,0.782029,0.233395,0.000022,0.734,495.600781,0.428154



Checkpoint: /content/drive/MyDrive/CFIS_Phishing_Benchmark_Final/03_Resultados_por_Fold/benchmark_principal_5fold.csv
Características: /content/drive/MyDrive/CFIS_Phishing_Benchmark_Final/04_Rankings_Estabilidade/benchmark_principal_features.jsonl


In [19]:

# ============================================================
# CFIS-V2 — B13 | ESTABILIDADE DA SELEÇÃO
# Jaccard entre folds + Índice de Nogueira
# Sem reexecutar seletores ou classificadores
# ============================================================

import json
from itertools import combinations

import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display

FEATURE_FILE = (
    PASTAS["rankings"] /
    "benchmark_principal_features.jsonl"
)

RESULT_FILE = (
    PASTAS["folds"] /
    "benchmark_principal_5fold.csv"
)

OUTPUT_FILE = (
    PASTAS["rankings"] /
    "b13_estabilidade_seletores.csv"
)

PAIR_FILE = (
    PASTAS["rankings"] /
    "b13_jaccard_entre_folds.csv"
)

MATRIX_FILE = (
    PASTAS["rankings"] /
    "b13_matrizes_jaccard.csv"
)

assert FEATURE_FILE.exists(), "Arquivo de características ausente."
assert RESULT_FILE.exists(), "Checkpoint do benchmark ausente."

# ------------------------------------------------------------
# 1. Carregar dados
# ------------------------------------------------------------

records = []

with open(FEATURE_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

features_df = pd.DataFrame(records)
results_df = pd.read_csv(RESULT_FILE)

# Manter somente combinações concluídas com sucesso
valid = results_df.loc[
    results_df["status"] == "OK",
    ["fold", "classifier", "method"]
].drop_duplicates()

# Deduplicar registros do JSONL, preservando o último
features_df = features_df.drop_duplicates(
    subset=["fold", "classifier", "method"],
    keep="last"
)

features_df = features_df.merge(
    valid,
    on=["fold", "classifier", "method"],
    how="inner"
)

# Universo original de características
p = int(Xdev.shape[1])

def jaccard(a, b):
    a, b = set(a), set(b)
    union = a | b
    return len(a & b) / len(union) if union else np.nan

def nogueira_index(feature_sets, p):
    """
    Índice de estabilidade de Nogueira.

    Z: matriz binária (execuções x características).
    Permite conjuntos de tamanhos diferentes.

    Usa variância amostral (ddof=1).
    """

    m = len(feature_sets)

    if m < 2:
        return np.nan

    universe = list(Xdev.columns)

    if len(universe) != p:
        raise ValueError("Universo inconsistente.")

    Z = np.array([
        [int(feature in set(selected)) for feature in universe]
        for selected in feature_sets
    ], dtype=float)

    k_bar = Z.sum(axis=1).mean()

    denominator = (k_bar / p) * (1 - k_bar / p)

    if denominator <= 0:
        # Seleção determinística de todas ou nenhuma feature
        return 1.0 if np.all(Z == Z[0]) else np.nan

    variance = Z.var(axis=0, ddof=1).mean()

    return 1.0 - variance / denominator

# ------------------------------------------------------------
# 2. Calcular estabilidade por classificador e método
# ------------------------------------------------------------

summary_rows = []
pair_rows = []
matrix_rows = []

for (classifier, method), group in features_df.groupby(
    ["classifier", "method"]
):

    group = group.sort_values("fold")

    folds = group["fold"].astype(int).tolist()
    feature_sets = [
        list(features) for features in group["features"]
    ]

    # Verificar duplicações e características inválidas
    universe = set(Xdev.columns)

    for fold, selected in zip(folds, feature_sets):
        if len(selected) != len(set(selected)):
            raise ValueError(
                f"Duplicatas: {classifier}/{method}/fold={fold}"
            )
        if not set(selected).issubset(universe):
            raise ValueError(
                f"Feature desconhecida: {classifier}/{method}/fold={fold}"
            )

    sizes = [len(s) for s in feature_sets]

    pair_values = []

    for i, j in combinations(range(len(folds)), 2):

        score = jaccard(feature_sets[i], feature_sets[j])
        pair_values.append(score)

        pair_rows.append({
            "classifier": classifier,
            "method": method,
            "fold_a": folds[i],
            "fold_b": folds[j],
            "jaccard": score
        })

    for i in range(len(folds)):
        for j in range(len(folds)):
            matrix_rows.append({
                "classifier": classifier,
                "method": method,
                "fold_a": folds[i],
                "fold_b": folds[j],
                "jaccard": jaccard(
                    feature_sets[i],
                    feature_sets[j]
                )
            })

    summary_rows.append({
        "classifier": classifier,
        "method": method,
        "n_folds": len(folds),
        "n_pairs": len(pair_values),
        "k_min": min(sizes),
        "k_max": max(sizes),
        "k_mean": np.mean(sizes),
        "jaccard_mean": (
            np.mean(pair_values) if pair_values else np.nan
        ),
        "jaccard_std": (
            np.std(pair_values, ddof=1)
            if len(pair_values) > 1 else np.nan
        ),
        "nogueira": nogueira_index(feature_sets, p)
    })

# ------------------------------------------------------------
# 3. Salvar resultados
# ------------------------------------------------------------

stability = pd.DataFrame(summary_rows)
pairs = pd.DataFrame(pair_rows)
matrices = pd.DataFrame(matrix_rows)

stability = stability.sort_values(
    ["classifier", "nogueira"],
    ascending=[True, False]
)

stability.to_csv(OUTPUT_FILE, index=False)
pairs.to_csv(PAIR_FILE, index=False)
matrices.to_csv(MATRIX_FILE, index=False)

# ------------------------------------------------------------
# 4. Exibir resultados
# ------------------------------------------------------------

print("=" * 70)
print("CFIS-V2 — ESTABILIDADE DOS SELETORES")
print("=" * 70)

print("Métodos/classificadores analisados:", len(stability))
print("Pares de folds avaliados:", len(pairs))

if not (stability["n_folds"] == 5).all():
    print(
        "ATENÇÃO: há combinações com menos de cinco folds."
    )

print("\nESTABILIDADE — JACCARD MÉDIO")
display(
    stability.pivot(
        index="method",
        columns="classifier",
        values="jaccard_mean"
    ).round(4)
)

print("\nESTABILIDADE — ÍNDICE DE NOGUEIRA")
display(
    stability.pivot(
        index="method",
        columns="classifier",
        values="nogueira"
    ).round(4)
)

print("\nRESUMO COMPLETO")
display(stability.round(4))

print("\nArquivos salvos:")
print(OUTPUT_FILE)
print(PAIR_FILE)
print(MATRIX_FILE)


CFIS-V2 — ESTABILIDADE DOS SELETORES
Métodos/classificadores analisados: 27
Pares de folds avaliados: 270

ESTABILIDADE — JACCARD MÉDIO


classifier,DecisionTree,LogReg,RandomForest
method,,,
All Features,1.0000,1.0000,1.0000
CFIS,0.7978,0.9389,0.9034
CFS,0.9125,0.9142,0.8412
L1,0.8419,0.8244,0.8603
MutualInfo,0.9000,0.9050,0.9276
PFI,0.3762,0.5981,0.4874
RandomTopK,0.0731,0.5223,0.3283
ReliefF,0.9000,0.9673,0.9696
TreeSHAP,0.8213,0.9514,0.9399



ESTABILIDADE — ÍNDICE DE NOGUEIRA


classifier,DecisionTree,LogReg,RandomForest
method,,,
All Features,1.0000,1.0000,1.0000
CFIS,0.8631,0.8980,0.8941
CFS,0.9436,0.8550,0.8205
L1,0.8953,0.6885,0.8435
MutualInfo,0.9356,0.8389,0.9217
PFI,0.4442,0.1730,0.2635
RandomTopK,-0.0472,-0.0150,-0.0495
ReliefF,0.9356,0.9463,0.9678
TreeSHAP,0.8792,0.9194,0.9356



RESUMO COMPLETO


,classifier,method,n_folds,n_pairs,k_min,k_max,k_mean,jaccard_mean,jaccard_std,nogueira
0,DecisionTree,All Features,5,10,87,87,87.0,1.0000,0.0000,1.0000
2,DecisionTree,CFS,5,10,15,15,15.0,0.9125,0.0604,0.9436
4,DecisionTree,MutualInfo,5,10,15,15,15.0,0.9000,0.0527,0.9356
7,DecisionTree,ReliefF,5,10,15,15,15.0,0.9000,0.0527,0.9356
3,DecisionTree,L1,5,10,15,15,15.0,0.8419,0.0533,0.8953
8,DecisionTree,TreeSHAP,5,10,15,15,15.0,0.8213,0.0815,0.8792
1,DecisionTree,CFIS,5,10,15,15,15.0,0.7978,0.0533,0.8631
5,DecisionTree,PFI,5,10,15,15,15.0,0.3762,0.0965,0.4442
6,DecisionTree,RandomTopK,5,10,15,15,15.0,0.0731,0.0451,-0.0472
9,LogReg,All Features,5,10,87,87,87.0,1.0000,0.0000,1.0000



Arquivos salvos:
/content/drive/MyDrive/CFIS_Phishing_Benchmark_Final/04_Rankings_Estabilidade/b13_estabilidade_seletores.csv
/content/drive/MyDrive/CFIS_Phishing_Benchmark_Final/04_Rankings_Estabilidade/b13_jaccard_entre_folds.csv
/content/drive/MyDrive/CFIS_Phishing_Benchmark_Final/04_Rankings_Estabilidade/b13_matrizes_jaccard.csv


In [20]:

# ============================================================
# CFIS-V2 — B14 | ANÁLISE ESTATÍSTICA
# Friedman + Wilcoxon + Holm + IC 95%
# Sem reexecutar o benchmark
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon, t
from statsmodels.stats.multitest import multipletests
from IPython.display import display

source = PASTAS["folds"] / "benchmark_principal_5fold.csv"
out = PASTAS["estatistica"]
out.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(source)
df = df[df["status"] == "OK"].copy()

methods = [
    "CFIS", "L1", "MutualInfo", "CFS",
    "ReliefF", "TreeSHAP", "PFI",
    "RandomTopK", "All Features"
]

metrics = [
    "macro_f1", "balanced_accuracy",
    "r2_labels", "rmse_labels"
]

friedman_rows = []
wilcoxon_rows = []
ci_rows = []

for classifier in ["DecisionTree", "LogReg", "RandomForest"]:

    subset = df[df["classifier"] == classifier]

    for metric in metrics:
        pivot = subset.pivot(
            index="fold", columns="method", values=metric
        ).reindex(columns=methods).sort_index()

        if pivot.shape != (5, len(methods)) or pivot.isna().any().any():
            raise ValueError(
                f"Dados incompletos: {classifier}/{metric}"
            )

        # Friedman: comparação global dos métodos
        statistic, pvalue = friedmanchisquare(
            *[pivot[m].values for m in methods]
        )

        friedman_rows.append({
            "classifier": classifier,
            "metric": metric,
            "n_folds": len(pivot),
            "n_methods": len(methods),
            "friedman_statistic": statistic,
            "p_value": pvalue
        })

        # Intervalos t de 95% das médias por método
        for method in methods:
            values = pivot[method].to_numpy(dtype=float)
            mean = values.mean()
            sd = values.std(ddof=1)
            margin = t.ppf(0.975, len(values)-1) * sd / np.sqrt(len(values))

            ci_rows.append({
                "classifier": classifier,
                "metric": metric,
                "method": method,
                "mean": mean,
                "std": sd,
                "ci95_lower": mean - margin,
                "ci95_upper": mean + margin
            })

        # Wilcoxon: CFIS contra cada comparador
        raw_tests = []

        for method in methods:
            if method == "CFIS":
                continue

            differences = (
                pivot["CFIS"] - pivot[method]
            ).to_numpy(dtype=float)

            if np.allclose(differences, 0, rtol=0, atol=1e-12):
                p = 1.0
                w = 0.0
            else:
                w, p = wilcoxon(
                    differences,
                    alternative="two-sided",
                    zero_method="wilcox",
                    method="auto"
                )

            raw_tests.append({
                "classifier": classifier,
                "metric": metric,
                "comparison": f"CFIS vs {method}",
                "mean_difference": differences.mean(),
                "wilcoxon_statistic": w,
                "p_raw": p
            })

        # Holm dentro da família de oito comparações
        adjusted = multipletests(
            [r["p_raw"] for r in raw_tests],
            alpha=0.05,
            method="holm"
        )

        for i, row in enumerate(raw_tests):
            row["p_holm"] = adjusted[1][i]
            row["reject_h0_05"] = bool(adjusted[0][i])
            wilcoxon_rows.append(row)

friedman_df = pd.DataFrame(friedman_rows)
wilcoxon_df = pd.DataFrame(wilcoxon_rows)
ci_df = pd.DataFrame(ci_rows)

friedman_df.to_csv(out / "b14_friedman.csv", index=False)
wilcoxon_df.to_csv(out / "b14_wilcoxon_holm.csv", index=False)
ci_df.to_csv(out / "b14_intervalos_confianca.csv", index=False)

print("FRIEDMAN — MACRO-F1")
display(
    friedman_df[
        friedman_df.metric == "macro_f1"
    ].round(6)
)

print("\nWILCOXON + HOLM — MACRO-F1")
display(
    wilcoxon_df[
        wilcoxon_df.metric == "macro_f1"
    ].round(6)
)

print("\nINTERVALOS DE CONFIANÇA — MACRO-F1")
display(
    ci_df[
        ci_df.metric == "macro_f1"
    ].round(6)
)

print("\nB14 concluída. Arquivos salvos em:", out)


FRIEDMAN — MACRO-F1


,classifier,metric,n_folds,n_methods,friedman_statistic,p_value
0,DecisionTree,macro_f1,5,9,26.400000,0.000897
4,LogReg,macro_f1,5,9,26.461538,0.000875
8,RandomForest,macro_f1,5,9,14.795987,0.063236



WILCOXON + HOLM — MACRO-F1


,classifier,metric,comparison,mean_difference,wilcoxon_statistic,p_raw,p_holm,reject_h0_05
0,DecisionTree,macro_f1,CFIS vs L1,0.001251,7.0,1.0000,1.000,False
1,DecisionTree,macro_f1,CFIS vs MutualInfo,0.014253,0.0,0.0625,0.500,False
2,DecisionTree,macro_f1,CFIS vs CFS,0.004878,3.0,0.3125,1.000,False
3,DecisionTree,macro_f1,CFIS vs ReliefF,0.010876,1.0,0.1250,0.625,False
4,DecisionTree,macro_f1,CFIS vs TreeSHAP,0.006625,0.0,0.0625,0.500,False
5,DecisionTree,macro_f1,CFIS vs PFI,0.003502,5.0,0.6250,1.000,False
6,DecisionTree,macro_f1,CFIS vs RandomTopK,0.106141,0.0,0.0625,0.500,False
7,DecisionTree,macro_f1,CFIS vs All Features,0.000877,7.0,1.0000,1.000,False
32,LogReg,macro_f1,CFIS vs L1,0.001500,0.0,0.0625,0.500,False
33,LogReg,macro_f1,CFIS vs MutualInfo,0.003000,0.0,0.1250,0.500,False



INTERVALOS DE CONFIANÇA — MACRO-F1


,classifier,metric,method,mean,std,ci95_lower,ci95_upper
0,DecisionTree,macro_f1,CFIS,0.930756,0.006123,0.923153,0.938359
1,DecisionTree,macro_f1,L1,0.929505,0.005079,0.923199,0.935811
2,DecisionTree,macro_f1,MutualInfo,0.916503,0.007241,0.907513,0.925494
3,DecisionTree,macro_f1,CFS,0.925878,0.009629,0.913922,0.937834
4,DecisionTree,macro_f1,ReliefF,0.919880,0.005424,0.913146,0.926614
5,DecisionTree,macro_f1,TreeSHAP,0.924132,0.005026,0.917891,0.930372
6,DecisionTree,macro_f1,PFI,0.927254,0.009127,0.915921,0.938587
7,DecisionTree,macro_f1,RandomTopK,0.824615,0.046945,0.766325,0.882906
8,DecisionTree,macro_f1,All Features,0.929879,0.004646,0.924110,0.935649
36,LogReg,macro_f1,CFIS,0.947131,0.001892,0.944781,0.949481



B14 concluída. Arquivos salvos em: /content/drive/MyDrive/CFIS_Phishing_Benchmark_Final/05_Eficiencia_Estatistica


In [21]:

# ============================================================
# CFIS-V2 — B15 | CONCORDÂNCIA JACCARD@K
# Comparações pareadas dentro do mesmo fold/classificador
# Sem reexecutar seletores ou treinamentos
# ============================================================

import json
import numpy as np
import pandas as pd
from IPython.display import display

source = PASTAS["rankings"] / "benchmark_principal_features.jsonl"
out_dir = PASTAS["rankings"]
out_dir.mkdir(parents=True, exist_ok=True)

records = []
with open(source, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)

required = {"fold", "classifier", "method", "features"}
assert required.issubset(df.columns), (
    f"Colunas ausentes: {required - set(df.columns)}"
)

df = df.drop_duplicates(
    ["fold", "classifier", "method"], keep="last"
).copy()

df["fold"] = df["fold"].astype(int)

# Apenas resultados confirmados como OK no benchmark
benchmark = pd.read_csv(
    PASTAS["folds"] / "benchmark_principal_5fold.csv"
)

ok = benchmark.loc[
    benchmark["status"].eq("OK"),
    ["fold", "classifier", "method"]
].drop_duplicates()

ok["fold"] = ok["fold"].astype(int)

df = df.merge(
    ok, on=["fold", "classifier", "method"], how="inner"
)

def jaccard_at_k(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else np.nan

rows = []

for (classifier, fold), group in df.groupby(
    ["classifier", "fold"]
):
    selected = dict(zip(group["method"], group["features"]))

    if "CFIS" not in selected:
        continue

    cfis = selected["CFIS"]

    for method, features in selected.items():
        if method in ("CFIS", "All Features"):
            continue

        if len(cfis) != len(features):
            raise ValueError(
                f"K diferente: {classifier}, fold {fold}, {method}"
            )

        rows.append({
            "classifier": classifier,
            "fold": fold,
            "method": method,
            "k": len(cfis),
            "intersection": len(set(cfis) & set(features)),
            "jaccard_at_k": jaccard_at_k(cfis, features)
        })

agreement = pd.DataFrame(rows)

summary = (
    agreement.groupby(["classifier", "method", "k"])
    .agg(
        folds=("fold", "nunique"),
        jaccard_mean=("jaccard_at_k", "mean"),
        jaccard_std=("jaccard_at_k", "std"),
        intersection_mean=("intersection", "mean")
    )
    .reset_index()
)

assert (summary["folds"] == 5).all(), (
    "Alguma comparação não possui cinco folds completos."
)

agreement.to_csv(
    out_dir / "b15_concordancia_jaccard_por_fold.csv",
    index=False
)

summary.to_csv(
    out_dir / "b15_concordancia_jaccard_resumo.csv",
    index=False
)

print("=" * 65)
print("CFIS-V2 — CONCORDÂNCIA JACCARD@K")
print("=" * 65)

print("\nCFIS versus métodos de referência:")
display(
    summary[
        summary["method"].isin(["TreeSHAP", "PFI"])
    ].round(4)
)

print("\nConcordância com todos os métodos:")
display(
    summary.pivot(
        index="method",
        columns="classifier",
        values="jaccard_mean"
    ).round(4)
)

print("\nArquivos salvos em:", out_dir)


CFIS-V2 — CONCORDÂNCIA JACCARD@K

CFIS versus métodos de referência:


,classifier,method,k,folds,jaccard_mean,jaccard_std,intersection_mean
3,DecisionTree,PFI,15,5,0.2866,0.0875,6.6
6,DecisionTree,TreeSHAP,15,5,0.3529,0.0517,7.8
10,LogReg,PFI,60,5,0.5867,0.0818,44.2
13,LogReg,TreeSHAP,60,5,0.7649,0.0184,52.0
17,RandomForest,PFI,45,5,0.5254,0.0954,30.8
20,RandomForest,TreeSHAP,45,5,0.6669,0.0218,36.0



Concordância com todos os métodos:


classifier,DecisionTree,LogReg,RandomForest
method,,,
CFS,0.4156,0.5588,0.4808
L1,0.6316,0.6577,0.5967
MutualInfo,0.2300,0.8241,0.7112
PFI,0.2866,0.5867,0.5254
RandomTopK,0.1062,0.5276,0.3457
ReliefF,0.2300,0.7752,0.6608
TreeSHAP,0.3529,0.7649,0.6669



Arquivos salvos em: /content/drive/MyDrive/CFIS_Phishing_Benchmark_Final/04_Rankings_Estabilidade


In [22]:

# ============================================================
# CFIS-V2 — B16A | AUDITORIA DOS RANKINGS DISPONÍVEIS
# Não executa seletores nem treina modelos
# ============================================================

import inspect
import json
import pandas as pd

print("=== SELETORES REGISTRADOS ===")

for name, fn in BENCHMARK_SELECTORS.items():
    print(f"\n{name}")
    print("Assinatura:", inspect.signature(fn))
    try:
        source = inspect.getsource(fn)
        print("Código (primeiros 2200 caracteres):")
        print(source[:2200])
    except (OSError, TypeError):
        print("Código-fonte indisponível.")

print("\n=== FUNÇÃO ORIGINAL CFIS ===")
print("Assinatura:", inspect.signature(cfis_fit))

print("\n=== ARQUIVOS DE RANKING EXISTENTES ===")
for path in sorted(PASTAS["rankings"].glob("*")):
    print(path.name, "|", path.stat().st_size, "bytes")

print("\n=== ESTRUTURA DO JSONL ===")
path = PASTAS["rankings"] / "benchmark_principal_features.jsonl"

with open(path, encoding="utf-8") as f:
    record = json.loads(next(line for line in f if line.strip()))

print("Campos:", list(record.keys()))
for key, value in record.items():
    if isinstance(value, list):
        print(f"{key}: lista com {len(value)} elementos")
    else:
        print(f"{key}: {str(value)[:150]}")


=== SELETORES REGISTRADOS ===

CFIS
Assinatura: (X_train, y_train, seed, k)
Código (primeiros 2200 caracteres):
def selector_cfis(X_train, y_train, seed, k):

    rank, cost = cfis_fit(
        X_train, y_train, seed=seed
    )

    return rank["feature"].tolist()[:k]


L1
Assinatura: (X_train, y_train, seed, k)
Código (primeiros 2200 caracteres):
def selector_l1(X_train, y_train, seed, k):

    model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=0.1,
        max_iter=1500,
        random_state=seed
    )

    model.fit(X_train, y_train)

    return ranking_from_scores(
        np.abs(model.coef_[0]),
        X_train.columns
    )[:k]


MutualInfo
Assinatura: (X_train, y_train, seed, k)
Código (primeiros 2200 caracteres):
def selector_mi(X_train, y_train, seed, k):

    scores = mutual_info_classif(
        X_train,
        y_train,
        discrete_features=benchmark_discrete_mask(X_train),
        random_state=seed
    )

    return ranking_from_sc

In [23]:

# ============================================================
# CFIS-V2 — B16B | CONCORDÂNCIA DE RANKINGS COMPLETOS
# CFIS x TreeSHAP x PFI
# 5 folds, rankings de 87 características
# Checkpoints individuais e validação contra B12.3
# ============================================================

import json
import time
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.stats import spearmanr, kendalltau
from IPython.display import display

METHODS = ["CFIS", "TreeSHAP", "PFI"]

RANK_DIR = PASTAS["rankings"] / "b16_rankings_completos"
RANK_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR = PASTAS["rankings"]

N_FEATURES = Xdev.shape[1]
FEATURE_NAMES = list(Xdev.columns)

K_VALUES = {
    "DecisionTree": 15,
    "LogReg": 60,
    "RandomForest": 45
}

# ------------------------------------------------------------
# 1. Carregar Top-K originais
# ------------------------------------------------------------

original_file = (
    PASTAS["rankings"] /
    "benchmark_principal_features.jsonl"
)

original_records = []

with open(original_file, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            original_records.append(json.loads(line))

original_df = pd.DataFrame(original_records)

original_df = original_df.drop_duplicates(
    ["fold", "classifier", "method"],
    keep="last"
)

original_lookup = {
    (int(row.fold), row.classifier, row.method):
        list(row.features)
    for row in original_df.itertuples()
}

# ------------------------------------------------------------
# 2. Recuperar rankings completos por fold
# ------------------------------------------------------------

all_rankings = {}
execution_rows = []

for fold_idx, (train_idx, val_idx) in enumerate(
    BENCHMARK_SPLITS, start=1
):

    print(f"\n{'=' * 65}")
    print(f"FOLD {fold_idx}/5")
    print("=" * 65)

    X_train_raw = Xdev.iloc[train_idx]
    y_train = ydev.iloc[train_idx]

    # Mesmo pré-processamento do benchmark
    X_train, _ = prepare(
        X_train_raw,
        Xdev.iloc[val_idx]
    )

    if list(X_train.columns) != FEATURE_NAMES:
        raise ValueError(
            f"Universo de features alterado no fold {fold_idx}"
        )

    for method in METHODS:

        checkpoint = (
            RANK_DIR /
            f"fold_{fold_idx}_{method}.json"
        )

        if checkpoint.exists():
            with open(checkpoint, encoding="utf-8") as f:
                record = json.load(f)

            ranking = record["ranking"]
            elapsed = record.get("selection_time", np.nan)
            print(f"  {method}: checkpoint reutilizado")

        else:
            print(f"  {method}: calculando...", flush=True)

            start = time.perf_counter()

            ranking = BENCHMARK_SELECTORS[method](
                X_train,
                y_train,
                seed=SEED,
                k=N_FEATURES
            )

            elapsed = time.perf_counter() - start

            record = {
                "fold": fold_idx,
                "method": method,
                "n_features": N_FEATURES,
                "selection_time": elapsed,
                "ranking": list(ranking)
            }

            temp = checkpoint.with_suffix(".tmp")
            with open(temp, "w", encoding="utf-8") as f:
                json.dump(record, f, ensure_ascii=False, indent=2)

            temp.replace(checkpoint)

            print(
                f"  {method}: {elapsed:.2f}s",
                flush=True
            )

        ranking = list(ranking)

        if (
            len(ranking) != N_FEATURES
            or len(set(ranking)) != N_FEATURES
            or set(ranking) != set(FEATURE_NAMES)
        ):
            raise ValueError(
                f"Ranking inválido: fold={fold_idx}, method={method}"
            )

        all_rankings[(fold_idx, method)] = ranking

        execution_rows.append({
            "fold": fold_idx,
            "method": method,
            "selection_time": elapsed,
            "n_features": len(ranking)
        })

# ------------------------------------------------------------
# 3. Validar rankings contra o benchmark original
# ------------------------------------------------------------

validation_rows = []

for fold in range(1, 6):
    for method in METHODS:

        ranking = all_rankings[(fold, method)]

        for classifier, k in K_VALUES.items():

            original = original_lookup.get(
                (fold, classifier, method)
            )

            if original is None:
                raise ValueError(
                    f"Top-K ausente: {fold}/{classifier}/{method}"
                )

            # A concordância é de conjuntos.
            # A B12.3 não salvou a ordem interna do Top-K.
            matches = set(ranking[:k]) == set(original)

            validation_rows.append({
                "fold": fold,
                "classifier": classifier,
                "method": method,
                "k": k,
                "matches_original": matches
            })

validation_df = pd.DataFrame(validation_rows)

validation_df.to_csv(
    OUT_DIR / "b16_validacao_rankings.csv",
    index=False
)

if not validation_df["matches_original"].all():
    print("\nDIVERGÊNCIAS COM O BENCHMARK ORIGINAL:")
    display(
        validation_df[
            ~validation_df["matches_original"]
        ]
    )
    raise ValueError(
        "Rankings não reproduzem os Top-K originais. "
        "Correlação interrompida para evitar resultados inválidos."
    )

print("\nValidação: todos os 45 Top-K reproduzidos.")

# ------------------------------------------------------------
# 4. Spearman, Kendall e Jaccard@K
# ------------------------------------------------------------

agreement_rows = []

for fold in range(1, 6):

    cfis_rank = all_rankings[(fold, "CFIS")]

    cfis_positions = {
        feature: position
        for position, feature in enumerate(cfis_rank)
    }

    for method in ["TreeSHAP", "PFI"]:

        other_rank = all_rankings[(fold, method)]

        other_positions = {
            feature: position
            for position, feature in enumerate(other_rank)
        }

        a = np.array([
            cfis_positions[f] for f in FEATURE_NAMES
        ])

        b = np.array([
            other_positions[f] for f in FEATURE_NAMES
        ])

        spearman = spearmanr(a, b).statistic
        kendall = kendalltau(a, b).statistic

        for classifier, k in K_VALUES.items():

            A = set(cfis_rank[:k])
            B = set(other_rank[:k])

            agreement_rows.append({
                "fold": fold,
                "classifier": classifier,
                "method": method,
                "k": k,
                "spearman": spearman,
                "kendall": kendall,
                "jaccard_at_k": len(A & B) / len(A | B),
                "intersection": len(A & B)
            })

agreement_df = pd.DataFrame(agreement_rows)

summary_df = (
    agreement_df.groupby(
        ["classifier", "method", "k"]
    )
    .agg(
        folds=("fold", "nunique"),
        spearman_mean=("spearman", "mean"),
        spearman_std=("spearman", "std"),
        kendall_mean=("kendall", "mean"),
        kendall_std=("kendall", "std"),
        jaccard_mean=("jaccard_at_k", "mean"),
        jaccard_std=("jaccard_at_k", "std")
    )
    .reset_index()
)

agreement_df.to_csv(
    OUT_DIR / "b16_correlacoes_por_fold.csv",
    index=False
)

summary_df.to_csv(
    OUT_DIR / "b16_correlacoes_resumo.csv",
    index=False
)

pd.DataFrame(execution_rows).to_csv(
    OUT_DIR / "b16_tempos_rankings.csv",
    index=False
)

print("\n" + "=" * 65)
print("CFIS-V2 — CONCORDÂNCIA DE RANKINGS COMPLETOS")
print("=" * 65)

display(summary_df.round(4))

print("\nB16B concluída.")
print("Rankings completos:", RANK_DIR)
print("Resumo:", OUT_DIR / "b16_correlacoes_resumo.csv")



FOLD 1/5
  CFIS: calculando...
  CFIS: 0.89s
  TreeSHAP: calculando...
  TreeSHAP: 27.09s
  PFI: calculando...
  PFI: 30.19s

FOLD 2/5
  CFIS: calculando...
  CFIS: 0.49s
  TreeSHAP: calculando...
  TreeSHAP: 21.64s
  PFI: calculando...
  PFI: 30.85s

FOLD 3/5
  CFIS: calculando...
  CFIS: 0.53s
  TreeSHAP: calculando...
  TreeSHAP: 20.52s
  PFI: calculando...
  PFI: 30.65s

FOLD 4/5
  CFIS: calculando...
  CFIS: 0.52s
  TreeSHAP: calculando...
  TreeSHAP: 19.35s
  PFI: calculando...
  PFI: 30.56s

FOLD 5/5
  CFIS: calculando...
  CFIS: 0.58s
  TreeSHAP: calculando...
  TreeSHAP: 20.63s
  PFI: calculando...
  PFI: 30.52s

DIVERGÊNCIAS COM O BENCHMARK ORIGINAL:


,fold,classifier,method,k,matches_original
11,2,RandomForest,CFIS,45,False
12,2,DecisionTree,TreeSHAP,15,False
13,2,LogReg,TreeSHAP,60,False
14,2,RandomForest,TreeSHAP,45,False
15,2,DecisionTree,PFI,15,False
16,2,LogReg,PFI,60,False
17,2,RandomForest,PFI,45,False
21,3,DecisionTree,TreeSHAP,15,False
22,3,LogReg,TreeSHAP,60,False
23,3,RandomForest,TreeSHAP,45,False


ValueError: Rankings não reproduzem os Top-K originais. Correlação interrompida para evitar resultados inválidos.

In [24]:

# ================================================================
# CFIS-V2 — B17 | HOLDOUT EXTERNO + TABELAS FINAIS DO ARTIGO
# Preserva CFIS original, K e hiperparâmetros do benchmark
# Checkpoints por método/classificador; sem retreinar CV
# ================================================================

import json, time, os, copy, gc, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import (
    f1_score, balanced_accuracy_score, accuracy_score,
    matthews_corrcoef, roc_auc_score,
    r2_score, mean_squared_error
)
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

ART = PASTAS["artigo"]
ART.mkdir(parents=True, exist_ok=True)

HOLD_DIR = PASTAS["folds"] / "b17_holdout"
HOLD_DIR.mkdir(parents=True, exist_ok=True)

METHODS = [
    "CFIS", "L1", "MutualInfo", "CFS",
    "ReliefF", "TreeSHAP", "PFI",
    "RandomTopK", "All Features"
]

CLASSIFIERS = ["DecisionTree", "LogReg", "RandomForest"]

# ---------------------------------------------------------------
# 1. Verificações de integridade
# ---------------------------------------------------------------

required_globals = [
    "Xdev", "ydev", "Xtest", "ytest", "prepare",
    "cfis_fit", "BENCHMARK_SELECTORS", "BENCHMARK_K",
    "models", "SEED", "PASTAS"
]

missing = [x for x in required_globals if x not in globals()]
if missing:
    raise RuntimeError(f"Objetos ausentes: {missing}")

assert len(Xdev) == 8001, "Tamanho inesperado do desenvolvimento."
assert len(Xtest) == 3429, "Tamanho inesperado do holdout."
assert Xdev.shape[1] == Xtest.shape[1] == 87
assert list(Xdev.columns) == list(Xtest.columns)

# Evitar qualquer sobreposição de índices originais quando
# os índices forem preservados na divisão do dataset.
if not Xdev.index.equals(pd.RangeIndex(len(Xdev))):
    overlap = Xdev.index.intersection(Xtest.index)
    if len(overlap):
        raise ValueError("Possível sobreposição de índices dev/holdout.")

# ---------------------------------------------------------------
# 2. Preparar os dados: ajuste apenas no desenvolvimento
# ---------------------------------------------------------------

X_train, X_hold = prepare(Xdev, Xtest)

if list(X_train.columns) != list(X_hold.columns):
    raise ValueError("Colunas incompatíveis após prepare().")

y_train = np.asarray(ydev).ravel()
y_hold = np.asarray(ytest).ravel()

assert len(y_train) == len(X_train)
assert len(y_hold) == len(X_hold)
assert set(np.unique(y_train)) == set(np.unique(y_hold))
assert len(np.unique(y_train)) == 2

# ---------------------------------------------------------------
# 3. Executar holdout com checkpoint por avaliação
# ---------------------------------------------------------------

def save_json_atomic(path, payload):
    temp = path.with_suffix(".tmp")
    with open(temp, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    os.replace(temp, path)

def new_model(name):
    model = models[name]
    return clone(model) if hasattr(model, "get_params") else model()

def evaluate_holdout(clf_name, method, k):

    checkpoint = HOLD_DIR / f"{clf_name}_{method.replace(' ', '_')}.json"

    if checkpoint.exists():
        with open(checkpoint, encoding="utf-8") as f:
            saved = json.load(f)
        if (
            saved.get("classifier") == clf_name
            and saved.get("method") == method
            and saved.get("k_protocol") == k
            and saved.get("n_holdout") == len(y_hold)
        ):
            print(f"REUTILIZADO | {clf_name:14s} | {method}")
            return saved

    t0 = time.perf_counter()

    if method == "All Features":
        selected = list(X_train.columns)
        selection_time = 0.0
    else:
        ts = time.perf_counter()
        selected = list(
            BENCHMARK_SELECTORS[method](
                X_train, y_train, seed=SEED, k=k
            )
        )
        selection_time = time.perf_counter() - ts

    if len(selected) != len(set(selected)):
        raise ValueError(f"Features duplicadas: {clf_name}/{method}")

    if len(selected) != (87 if method == "All Features" else k):
        raise ValueError(f"K incorreto: {clf_name}/{method}")

    if not set(selected).issubset(set(X_train.columns)):
        raise ValueError(f"Features desconhecidas: {clf_name}/{method}")

    model = new_model(clf_name)

    tf = time.perf_counter()
    model.fit(X_train[selected], y_train)
    fit_time = time.perf_counter() - tf

    tp = time.perf_counter()
    pred = model.predict(X_hold[selected])
    prediction_time = time.perf_counter() - tp

    auc = np.nan
    try:
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_hold[selected])[:, 1]
            auc = roc_auc_score(y_hold, proba)
        elif hasattr(model, "decision_function"):
            scores = model.decision_function(X_hold[selected])
            auc = roc_auc_score(y_hold, scores)
    except (ValueError, IndexError):
        pass

    record = {
        "classifier": clf_name,
        "method": method,
        "k_protocol": int(k),
        "k_selected": len(selected),
        "n_development": len(y_train),
        "n_holdout": len(y_hold),
        "macro_f1": float(f1_score(y_hold, pred, average="macro")),
        "balanced_accuracy": float(balanced_accuracy_score(y_hold, pred)),
        "accuracy": float(accuracy_score(y_hold, pred)),
        "mcc": float(matthews_corrcoef(y_hold, pred)),
        "roc_auc": float(auc),
        "r2": float(r2_score(y_hold, pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_hold, pred))),
        "selection_time": float(selection_time),
        "fit_time": float(fit_time),
        "prediction_time": float(prediction_time),
        "total_time": float(time.perf_counter() - t0),
        "features": selected,
        "seed": int(SEED)
    }

    save_json_atomic(checkpoint, record)

    print(
        f"OK | {clf_name:14s} | {method:13s} "
        f"| F1={record['macro_f1']:.4f} "
        f"| seleção={selection_time:.2f}s"
    )

    return record

holdout_records = []

for classifier in CLASSIFIERS:
    k = int(BENCHMARK_K[classifier])

    for method in METHODS:
        holdout_records.append(
            evaluate_holdout(classifier, method, k)
        )
        gc.collect()

holdout_df = pd.DataFrame(holdout_records)

assert len(holdout_df) == 27

# Remover lista de features da tabela numérica;
# ela continua preservada nos checkpoints JSON.
holdout_table = holdout_df.drop(columns=["features"])

holdout_table.to_csv(
    ART / "tabela_01_holdout_externo.csv",
    index=False
)

# ---------------------------------------------------------------
# 4. Consolidar resultados do benchmark de validação cruzada
# ---------------------------------------------------------------

cv = pd.read_csv(
    PASTAS["folds"] / "benchmark_principal_5fold.csv"
)

cv = cv[cv["status"] == "OK"].copy()

cv_summary = (
    cv.groupby(["classifier", "method"])
    .agg(
        cv_macro_f1_mean=("macro_f1", "mean"),
        cv_macro_f1_std=("macro_f1", "std"),
        cv_bal_acc_mean=("balanced_accuracy", "mean"),
        cv_selection_time_mean=("selection_time", "mean"),
        cv_total_time_mean=("total_time", "mean"),
        cv_folds=("fold", "nunique")
    )
    .reset_index()
)

assert len(cv_summary) == 27
assert (cv_summary.cv_folds == 5).all()

# ---------------------------------------------------------------
# 5. Integrar estabilidade
# ---------------------------------------------------------------

stability = pd.read_csv(
    PASTAS["rankings"] / "b13_estabilidade_seletores.csv"
)

stability = stability[
    ["classifier", "method", "jaccard_mean", "nogueira"]
].rename(columns={
    "jaccard_mean": "stability_jaccard",
    "nogueira": "stability_nogueira"
})

# ---------------------------------------------------------------
# 6. Integrar concordância CFIS x métodos concorrentes
# ---------------------------------------------------------------

agreement = pd.read_csv(
    PASTAS["rankings"] / "b15_concordancia_jaccard_resumo.csv"
)

agreement = agreement[
    ["classifier", "method", "jaccard_mean"]
].rename(columns={
    "jaccard_mean": "agreement_with_cfis_jaccard"
})

# ---------------------------------------------------------------
# 7. Tabela mestra do artigo
# ---------------------------------------------------------------

master = (
    cv_summary
    .merge(
        holdout_table,
        on=["classifier", "method"],
        how="inner",
        validate="one_to_one"
    )
    .merge(
        stability,
        on=["classifier", "method"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        agreement,
        on=["classifier", "method"],
        how="left",
        validate="one_to_one"
    )
)

master.loc[
    master.method == "CFIS",
    "agreement_with_cfis_jaccard"
] = 1.0

master = master.rename(columns={
    "macro_f1": "holdout_macro_f1",
    "balanced_accuracy": "holdout_bal_acc",
    "accuracy": "holdout_accuracy",
    "mcc": "holdout_mcc",
    "roc_auc": "holdout_roc_auc",
    "r2": "holdout_r2",
    "rmse": "holdout_rmse",
    "selection_time": "holdout_selection_time",
    "fit_time": "holdout_fit_time",
    "prediction_time": "holdout_prediction_time",
    "total_time": "holdout_total_time"
})

master = master.sort_values(["classifier", "method"])

master.to_csv(
    ART / "CFIS_V2_TABELA_MESTRA_ARTIGO.csv",
    index=False
)

# ---------------------------------------------------------------
# 8. Tabela compacta para compartilhar no ChatGPT
# ---------------------------------------------------------------

compact_cols = [
    "classifier", "method", "k_selected",
    "cv_macro_f1_mean", "cv_macro_f1_std",
    "holdout_macro_f1", "holdout_bal_acc",
    "holdout_roc_auc", "holdout_mcc",
    "cv_selection_time_mean",
    "holdout_selection_time",
    "stability_nogueira",
    "agreement_with_cfis_jaccard"
]

compact = master[compact_cols].copy()

compact.to_csv(
    ART / "CFIS_V2_TABELA_FINAL_COMPARTILHAR.csv",
    index=False
)

# ---------------------------------------------------------------
# 9. Incorporar testes estatísticos e correlações, se existirem
# ---------------------------------------------------------------

extra_files = {
    "friedman": PASTAS["estatistica"] / "b14_friedman.csv",
    "wilcoxon_holm": PASTAS["estatistica"] / "b14_wilcoxon_holm.csv",
    "intervalos_confianca": PASTAS["estatistica"] / "b14_intervalos_confianca.csv",
    "spearman_kendall": PASTAS["rankings"] / "b16_correlacoes_resumo.csv"
}

# Pacote em Excel com abas separadas, caso openpyxl esteja instalado
excel_path = ART / "CFIS_V2_RESULTADOS_COMPLETOS_ARTIGO.xlsx"

try:
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        compact.to_excel(writer, sheet_name="Resumo_Final", index=False)
        master.to_excel(writer, sheet_name="Tabela_Mestra", index=False)
        cv_summary.to_excel(writer, sheet_name="Validacao_Cruzada", index=False)
        holdout_table.to_excel(writer, sheet_name="Holdout_Externo", index=False)
        stability.to_excel(writer, sheet_name="Estabilidade", index=False)
        agreement.to_excel(writer, sheet_name="Concordancia_TopK", index=False)

        for sheet, path in extra_files.items():
            if path.exists():
                pd.read_csv(path).to_excel(
                    writer,
                    sheet_name=sheet[:31],
                    index=False
                )

    print("\nExcel criado:", excel_path)
except ImportError:
    print("\nExcel não criado: openpyxl indisponível.")

# ---------------------------------------------------------------
# 10. Exibição final
# ---------------------------------------------------------------

print("\n" + "=" * 78)
print("CFIS-V2 — TABELA FINAL PARA REDAÇÃO DO ARTIGO")
print("=" * 78)

display(compact.round(4))

print("\nHOLDOUT — MACRO-F1")
display(
    master.pivot(
        index="method",
        columns="classifier",
        values="holdout_macro_f1"
    ).round(4)
)

print("\nARQUIVOS FINAIS:")
print(ART / "CFIS_V2_TABELA_MESTRA_ARTIGO.csv")
print(ART / "CFIS_V2_TABELA_FINAL_COMPARTILHAR.csv")
print(excel_path)

print("\nB17 concluída.")


OK | DecisionTree   | CFIS          | F1=0.9355 | seleção=1.64s
OK | DecisionTree   | L1            | F1=0.9355 | seleção=0.20s
OK | DecisionTree   | MutualInfo    | F1=0.9309 | seleção=4.39s
OK | DecisionTree   | CFS           | F1=0.9355 | seleção=0.39s
OK | DecisionTree   | ReliefF       | F1=0.9204 | seleção=100.11s
OK | DecisionTree   | TreeSHAP      | F1=0.9329 | seleção=25.90s
OK | DecisionTree   | PFI           | F1=0.9326 | seleção=35.06s
OK | DecisionTree   | RandomTopK    | F1=0.8309 | seleção=0.00s
OK | DecisionTree   | All Features  | F1=0.9379 | seleção=0.00s
OK | LogReg         | CFIS          | F1=0.9425 | seleção=0.65s
OK | LogReg         | L1            | F1=0.9405 | seleção=0.12s
OK | LogReg         | MutualInfo    | F1=0.9402 | seleção=2.76s
OK | LogReg         | CFS           | F1=0.9373 | seleção=0.07s
OK | LogReg         | ReliefF       | F1=0.9396 | seleção=93.63s
OK | LogReg         | TreeSHAP      | F1=0.9399 | seleção=25.27s
OK | LogReg         | PFI         

KeyError: "Column(s) ['selection_time', 'total_time'] do not exist"

In [25]:

# ============================================================
# CFIS-V2 — B17.1 | CONSOLIDAÇÃO FINAL CORRIGIDA
# NÃO EXECUTA SELETORES, TREINAMENTO OU HOLDOUT
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

ART = PASTAS["artigo"]
ART.mkdir(parents=True, exist_ok=True)

HOLD_DIR = PASTAS["folds"] / "b17_holdout"

METHODS = [
    "CFIS", "L1", "MutualInfo", "CFS",
    "ReliefF", "TreeSHAP", "PFI",
    "RandomTopK", "All Features"
]
CLASSIFIERS = ["DecisionTree", "LogReg", "RandomForest"]

# 1. Carregar exclusivamente os checkpoints existentes
records = []

for clf in CLASSIFIERS:
    for method in METHODS:
        path = HOLD_DIR / f"{clf}_{method.replace(' ', '_')}.json"
        if not path.exists():
            raise FileNotFoundError(f"Checkpoint ausente: {path}")
        with open(path, encoding="utf-8") as f:
            records.append(json.load(f))

hold = pd.DataFrame(records)
assert len(hold) == 27

# 2. Carregar validação cruzada já concluída
cv = pd.read_csv(
    PASTAS["folds"] / "benchmark_principal_5fold.csv"
)
cv = cv[cv["status"].astype(str).str.upper() == "OK"].copy()

# Resolver nomes reais de colunas, sem alterar dados
def locate_column(frame, candidates, required=True):
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate
    if required:
        raise KeyError(
            f"Nenhuma coluna encontrada entre {candidates}. "
            f"Colunas disponíveis: {list(frame.columns)}"
        )
    return None

selection_col = locate_column(
    cv,
    ["selection_time", "selection_time_s",
     "selector_time", "selector_time_s",
     "time_selection", "tempo_selecao",
     "tempo_selecao_s"],
    required=False
)

total_col = locate_column(
    cv,
    ["total_time", "total_time_s",
     "elapsed_time", "elapsed_time_s",
     "time_total", "tempo_total",
     "tempo_total_s"],
    required=False
)

# Outras métricas obrigatórias
f1_col = locate_column(cv, ["macro_f1", "f1_macro"])
ba_col = locate_column(
    cv, ["balanced_accuracy", "balanced_acc", "bal_acc"]
)

print("COLUNAS RECONHECIDAS:")
print("Macro-F1:", f1_col)
print("Balanced Accuracy:", ba_col)
print("Tempo de seleção:", selection_col)
print("Tempo total:", total_col)

# 3. Agregar CV de forma segura
aggregations = {
    "cv_macro_f1_mean": (f1_col, "mean"),
    "cv_macro_f1_std": (f1_col, "std"),
    "cv_bal_acc_mean": (ba_col, "mean"),
    "cv_folds": ("fold", "nunique")
}

if selection_col:
    aggregations["cv_selection_time_mean"] = (
        selection_col, "mean"
    )

if total_col:
    aggregations["cv_total_time_mean"] = (
        total_col, "mean"
    )

cv_summary = (
    cv.groupby(["classifier", "method"])
    .agg(**aggregations)
    .reset_index()
)

assert len(cv_summary) == 27
assert (cv_summary["cv_folds"] == 5).all()

# 4. Carregar estabilidade e concordância
stability = pd.read_csv(
    PASTAS["rankings"] / "b13_estabilidade_seletores.csv"
)

stability = stability[
    ["classifier", "method", "jaccard_mean", "nogueira"]
].rename(columns={
    "jaccard_mean": "stability_jaccard",
    "nogueira": "stability_nogueira"
})

agreement = pd.read_csv(
    PASTAS["rankings"] / "b15_concordancia_jaccard_resumo.csv"
)

agreement = agreement[
    ["classifier", "method", "jaccard_mean"]
].rename(columns={
    "jaccard_mean": "agreement_with_cfis_jaccard"
})

# 5. Preparar tabela holdout
hold_table = hold.drop(columns=["features"], errors="ignore")

hold_table = hold_table.rename(columns={
    "macro_f1": "holdout_macro_f1",
    "balanced_accuracy": "holdout_bal_acc",
    "accuracy": "holdout_accuracy",
    "mcc": "holdout_mcc",
    "roc_auc": "holdout_roc_auc",
    "r2": "holdout_r2",
    "rmse": "holdout_rmse",
    "selection_time": "holdout_selection_time",
    "fit_time": "holdout_fit_time",
    "prediction_time": "holdout_prediction_time",
    "total_time": "holdout_total_time"
})

# 6. Consolidar
master = (
    cv_summary
    .merge(
        hold_table,
        on=["classifier", "method"],
        validate="one_to_one"
    )
    .merge(
        stability,
        on=["classifier", "method"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        agreement,
        on=["classifier", "method"],
        how="left",
        validate="one_to_one"
    )
)

master.loc[
    master["method"] == "CFIS",
    "agreement_with_cfis_jaccard"
] = 1.0

master = master.sort_values(
    ["classifier", "method"]
).reset_index(drop=True)

assert len(master) == 27

# 7. Arquivos CSV finais
master_path = ART / "CFIS_V2_TABELA_MESTRA_ARTIGO.csv"
master.to_csv(master_path, index=False)

compact_cols = [
    "classifier", "method", "k_selected",
    "cv_macro_f1_mean", "cv_macro_f1_std",
    "holdout_macro_f1", "holdout_bal_acc",
    "holdout_roc_auc", "holdout_mcc",
    "stability_nogueira",
    "stability_jaccard",
    "agreement_with_cfis_jaccard",
    "holdout_selection_time"
]

if "cv_selection_time_mean" in master.columns:
    compact_cols.append("cv_selection_time_mean")

compact = master[compact_cols]

compact_path = (
    ART / "CFIS_V2_TABELA_FINAL_COMPARTILHAR.csv"
)
compact.to_csv(compact_path, index=False)

# 8. Excel completo para o artigo
excel_path = (
    ART / "CFIS_V2_RESULTADOS_COMPLETOS_ARTIGO.xlsx"
)

extra_files = {
    "Friedman": PASTAS["estatistica"] / "b14_friedman.csv",
    "Wilcoxon_Holm": PASTAS["estatistica"] / "b14_wilcoxon_holm.csv",
    "IC_95": PASTAS["estatistica"] / "b14_intervalos_confianca.csv",
    "Spearman_Kendall": (
        PASTAS["rankings"] / "b16_correlacoes_resumo.csv"
    )
}

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    compact.to_excel(writer, "Resumo_Final", index=False)
    master.to_excel(writer, "Tabela_Mestra", index=False)
    cv_summary.to_excel(writer, "Validacao_Cruzada", index=False)
    hold_table.to_excel(writer, "Holdout_Externo", index=False)
    stability.to_excel(writer, "Estabilidade", index=False)
    agreement.to_excel(writer, "Concordancia_TopK", index=False)

    for sheet, path in extra_files.items():
        if path.exists():
            pd.read_csv(path).to_excel(
                writer, sheet_name=sheet, index=False
            )

print("\n" + "=" * 72)
print("CFIS-V2 — CONSOLIDAÇÃO FINAL CONCLUÍDA")
print("=" * 72)

display(compact.round(4))

print("\nARQUIVOS GERADOS:")
print(master_path)
print(compact_path)
print(excel_path)
print("\nNenhum experimento foi reexecutado.")


COLUNAS RECONHECIDAS:
Macro-F1: macro_f1
Balanced Accuracy: balanced_accuracy
Tempo de seleção: selection_time_s
Tempo total: total_time_s

CFIS-V2 — CONSOLIDAÇÃO FINAL CONCLUÍDA


,classifier,method,k_selected,cv_macro_f1_mean,cv_macro_f1_std,holdout_macro_f1,holdout_bal_acc,holdout_roc_auc,holdout_mcc,stability_nogueira,stability_jaccard,agreement_with_cfis_jaccard,holdout_selection_time,cv_selection_time_mean
0,DecisionTree,All Features,87,0.9299,0.0046,0.9379,0.9379,0.9379,0.8759,1.0000,1.0000,NaN,0.0000,0.0000
1,DecisionTree,CFIS,15,0.9308,0.0061,0.9355,0.9356,0.9356,0.8711,0.8631,0.7978,1.0000,1.6403,0.7446
2,DecisionTree,CFS,15,0.9259,0.0096,0.9355,0.9356,0.9356,0.8712,0.9436,0.9125,0.4156,0.3937,0.1344
3,DecisionTree,L1,15,0.9295,0.0051,0.9355,0.9356,0.9356,0.8712,0.8953,0.8419,0.6316,0.2016,0.1415
4,DecisionTree,MutualInfo,15,0.9165,0.0072,0.9309,0.9309,0.9309,0.8618,0.9356,0.9000,0.2300,4.3901,2.3379
5,DecisionTree,PFI,15,0.9273,0.0091,0.9326,0.9326,0.9326,0.8653,0.4442,0.3762,0.2866,35.0609,30.7506
6,DecisionTree,RandomTopK,15,0.8246,0.0469,0.8309,0.8309,0.8326,0.6617,-0.0472,0.0731,0.1062,0.0003,0.0003
7,DecisionTree,ReliefF,15,0.9199,0.0054,0.9204,0.9204,0.9204,0.8408,0.9356,0.9000,0.2300,100.1119,65.3576
8,DecisionTree,TreeSHAP,15,0.9241,0.0050,0.9329,0.9329,0.9329,0.8660,0.8792,0.8213,0.3529,25.8975,19.8431
9,LogReg,All Features,87,0.9455,0.0023,0.9396,0.9396,0.9842,0.8793,1.0000,1.0000,NaN,0.0000,0.0000



ARQUIVOS GERADOS:
/content/drive/MyDrive/CFIS_Phishing_Benchmark_Final/06_Tabelas_Graficos_Artigo/CFIS_V2_TABELA_MESTRA_ARTIGO.csv
/content/drive/MyDrive/CFIS_Phishing_Benchmark_Final/06_Tabelas_Graficos_Artigo/CFIS_V2_TABELA_FINAL_COMPARTILHAR.csv
/content/drive/MyDrive/CFIS_Phishing_Benchmark_Final/06_Tabelas_Graficos_Artigo/CFIS_V2_RESULTADOS_COMPLETOS_ARTIGO.xlsx

Nenhum experimento foi reexecutado.


In [26]:

from pathlib import Path
from google.colab import files

arquivo = Path(
    "/content/drive/MyDrive/CFIS_Phishing_Benchmark_Final/"
    "06_Tabelas_Graficos_Artigo/"
    "CFIS_V2_RESULTADOS_COMPLETOS_ARTIGO.xlsx"
)

if arquivo.is_file():
    print(f"Baixando: {arquivo.name}")
    files.download(str(arquivo))
else:
    print(f"Arquivo não encontrado: {arquivo}")



Baixando: CFIS_V2_RESULTADOS_COMPLETOS_ARTIGO.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>